In [ ]:
import pandas as pd
import sqlite3
import os

# making the database connection

db_path = "../data/patents.db"
conn = sqlite3.connect(db_path)

print("connected to db:", db_path)

connected to db: ../data/patents.db


In [2]:
# loading cleaned csv files in chucks to avoid memory issues since the files are big.
# this is much faster than loading the whole file into pandas memory and then pushing to sqlite
def load_csv_to_sqlite_in_chunks(csv_path, table_name, conn, chunk_size=50000):
    first_chunk = True
    total_rows = 0

    for chunk in pd.read_csv(csv_path, chunksize=chunk_size, low_memory=False):
        chunk.to_sql(
            table_name,
            conn,
            if_exists="replace" if first_chunk else "append",
            index=False
        )

        total_rows += len(chunk)
        first_chunk = False
        print(f"{table_name}: loaded {total_rows:,} rows")

    print(f"done loading {table_name}\n")

# loading the cleaned csv files in chunks
load_csv_to_sqlite_in_chunks("../data/clean_patents.csv", "patents", conn)
load_csv_to_sqlite_in_chunks("../data/clean_inventors.csv", "inventors", conn)
load_csv_to_sqlite_in_chunks("../data/clean_companies.csv", "companies", conn)
load_csv_to_sqlite_in_chunks("../data/clean_relationships.csv", "relationships", conn)

print("all clean csv files loaded into sqlite okay")

conn.close()

patents: loaded 50,000 rows
patents: loaded 100,000 rows
patents: loaded 150,000 rows
patents: loaded 200,000 rows
patents: loaded 250,000 rows
patents: loaded 300,000 rows
patents: loaded 350,000 rows
patents: loaded 400,000 rows
patents: loaded 450,000 rows
patents: loaded 500,000 rows
patents: loaded 550,000 rows
patents: loaded 600,000 rows
patents: loaded 650,000 rows
patents: loaded 700,000 rows
patents: loaded 750,000 rows
patents: loaded 800,000 rows
patents: loaded 850,000 rows
patents: loaded 900,000 rows
patents: loaded 950,000 rows
patents: loaded 1,000,000 rows
patents: loaded 1,050,000 rows
patents: loaded 1,100,000 rows
patents: loaded 1,150,000 rows
patents: loaded 1,200,000 rows
patents: loaded 1,250,000 rows
patents: loaded 1,300,000 rows
patents: loaded 1,350,000 rows
patents: loaded 1,400,000 rows
patents: loaded 1,450,000 rows
patents: loaded 1,500,000 rows
patents: loaded 1,550,000 rows
patents: loaded 1,600,000 rows
patents: loaded 1,650,000 rows
patents: loaded 

In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/patents.db")

for table in ["patents", "inventors", "companies", "relationships"]:
    df = pd.read_sql(f"SELECT COUNT(*) AS total FROM {table}", conn)
    print(table, "rows:", df.loc[0, "total"])

conn.close()

patents rows: 9454161
inventors rows: 4294034
companies rows: 516032
relationships rows: 25291173


In [4]:
import os
import sqlite3
import pandas as pd
import tempfile

# forcing python temp files to use D drive 
temp_dir = r"D:\patent_pipeline\temp"
os.makedirs(temp_dir, exist_ok=True)

os.environ["TMP"] = temp_dir
os.environ["TEMP"] = temp_dir
tempfile.tempdir = temp_dir

print("temp files will go here:", temp_dir)

# now connect to the database
conn = sqlite3.connect(r"D:\patent_pipeline\data\patents.db")

# a few sqlite settings to reduce disk pressure a bit
conn.execute("PRAGMA temp_store = FILE;")
conn.execute(f"PRAGMA temp_store_directory = '{temp_dir.replace('\\', '\\\\')}';")
conn.execute("PRAGMA cache_size = 10000;")

print("connected okay")

temp files will go here: D:\patent_pipeline\temp
connected okay


In [5]:
#trends over time - how many patents per year?
q = """
SELECT year, COUNT(1) as total
FROM patents
WHERE year IS NOT NULL
GROUP BY year
ORDER BY year;
"""

trends = pd.read_sql(q, conn)
print(trends)

#save it
trends.to_csv("../output/patent_trends.csv", index=False)


    year   total
0   1976   70941
1   1977   69820
2   1978   70586
3   1979   52484
4   1980   66212
5   1981   71112
6   1982   63292
7   1983   62019
8   1984   72689
9   1985   77276
10  1986   77050
11  1987   89591
12  1988   84449
13  1989  102719
14  1990   99275
15  1991  106891
16  1992  107540
17  1993  109933
18  1994  113723
19  1995  113954
20  1996  121811
21  1997  124181
22  1998  163265
23  1999  169252
24  2000  176192
25  2001  184172
26  2002  184494
27  2003  187104
28  2004  181413
29  2005  157829
30  2006  196489
31  2007  182978
32  2008  185260
33  2009  192052
34  2010  244599
35  2011  248101
36  2012  277285
37  2013  303658
38  2014  327014
39  2015  326969
40  2016  334674
41  2017  352587
42  2018  341104
43  2019  392618
44  2020  390572
45  2021  363829
46  2022  360417
47  2023  350093
48  2024  373852
49  2025  378741


In [6]:
#top countries by patent count
q = """
SELECT country, COUNT(1) as total
FROM inventors
WHERE country IS NOT NULL
GROUP BY country
ORDER BY total DESC
LIMIT 10;
"""

top_countries = pd.read_sql(q, conn)
print(top_countries)

top_countries.to_csv("../output/top_countries.csv", index=False)

  country    total
0      US  1987483
1      JP   499038
2      DE   302484
3      CN   196441
4      KR   154040
5      FR   142006
6      CA   113916
7      GB   112701
8      TW   100718
9      IN    74851


In [7]:
#top inventors by patent count
q = """
SELECT i.name, t.total_patents
FROM (
    SELECT inventor_id, COUNT(DISTINCT patent_id) as total_patents
    FROM relationships
    GROUP BY inventor_id
    ORDER BY total_patents DESC
    LIMIT 10
) t
JOIN inventors i
ON t.inventor_id = i.inventor_id;
"""

top_inventors = pd.read_sql(q, conn)
print(top_inventors)
top_inventors.to_csv("../output/top_inventors.csv", index=False)

                       name  total_patents
0           Kia Silverbrook           4778
1          Shunpei Yamazaki           6787
2          Bartley K. Andre           2478
3  Frederick E. Shelton, IV           2723
4        Duncan Robert Kerr           2534
5             Kangguo Cheng           2835
6                   Tao Luo           4490
7                Peter Gaal           2554
8                  Junyi Li           2881
9           Jonathan P. Ive           2947


In [8]:
#top companies by patent count
q = """
SELECT c.name, t.total_patents
FROM (
    SELECT company_id, COUNT(DISTINCT patent_id) as total_patents
    FROM relationships
    WHERE company_id IS NOT NULL
    GROUP BY company_id
    ORDER BY total_patents DESC
    LIMIT 10
) t
JOIN companies c
ON t.company_id = c.company_id;
"""

top_companies = pd.read_sql(q, conn)
print(top_companies)
top_companies.to_csv("../output/top_companies.csv", index=False)



                                          name  total_patents
0                       CANON KABUSHIKI KAISHA          91328
1                                HITACHI, LTD.          46073
2                            Intel Corporation          53218
3                          LG ELECTRONICS INC.          50000
4                       SONY GROUP CORPORATION          62908
5  International Business Machines Corporation         164081
6                    SAMSUNG DISPLAY CO., LTD.         174536
7                     General Electric Company          52040
8                              Fujitsu Limited          56341
9                     Kabushiki Kaisha Toshiba          53604


In [9]:
    #join querry
q = """
SELECT p.patent_id, p.title, i.name AS inventor_name, c.name AS company_name, p.year
FROM relationships r
JOIN patents p ON r.patent_id = p.patent_id
LEFT JOIN inventors i ON r.inventor_id = i.inventor_id
LEFT JOIN companies c ON r.company_id = c.company_id
LIMIT 20;
"""

join_result = pd.read_sql(q, conn)
print(join_result)

   patent_id                                              title  \
0   D1006496                                             Pillow   
1   12029253  Method for regulating the vaporisation of a va...   
2    6584128  Thermoelectric cooler driver utilizing unipola...   
3    4789863                  Pay per view entertainment system   
4   11161990  Voided latex particles containing functionaliz...   
5    6795487                                           Receiver   
6   12324454     Temperature control for a rotary head extruder   
7    D474886                                  Earplug container   
8   11885288                       Fuel pump module for vehicle   
9   11885288                       Fuel pump module for vehicle   
10  11885288                       Fuel pump module for vehicle   
11   7646155                       Generic motor control system   
12   4339721                            Electrostatic voltmeter   
13   6610738              N-disubstituted carbamoyloxy flavone

In [10]:
#cte querry
q = """
WITH inventor_counts AS (
    SELECT inventor_id, COUNT(DISTINCT patent_id) AS total_patents
    FROM relationships
    GROUP BY inventor_id
)
SELECT i.name, ic.total_patents
FROM inventor_counts ic
JOIN inventors i ON ic.inventor_id = i.inventor_id
ORDER BY ic.total_patents DESC
LIMIT 10;
"""

cte_result = pd.read_sql(q, conn)
print(cte_result)

                       name  total_patents
0          Shunpei Yamazaki           6787
1           Kia Silverbrook           4778
2                   Tao Luo           4490
3           Jonathan P. Ive           2947
4                  Junyi Li           2881
5             Kangguo Cheng           2835
6  Frederick E. Shelton, IV           2723
7                Peter Gaal           2554
8        Duncan Robert Kerr           2534
9          Bartley K. Andre           2478


In [11]:
#ranking querry
q = """
SELECT name, total_patents,
       RANK() OVER (ORDER BY total_patents DESC) AS inventor_rank
FROM (
    SELECT i.name, COUNT(DISTINCT r.patent_id) AS total_patents
    FROM relationships r
    JOIN inventors i ON r.inventor_id = i.inventor_id
    GROUP BY i.name
)
LIMIT 10;
"""

ranking_result = pd.read_sql(q, conn)
print(ranking_result)


                       name  total_patents  inventor_rank
0          Shunpei Yamazaki           6787              1
1           Kia Silverbrook           4778              2
2                   Tao Luo           4490              3
3           Jonathan P. Ive           2955              4
4                  Junyi Li           2883              5
5             Kangguo Cheng           2835              6
6  Frederick E. Shelton, IV           2723              7
7                Peter Gaal           2554              8
8        Duncan Robert Kerr           2534              9
9          Bartley K. Andre           2478             10


In [12]:
# country trends - patents by country over time
q = """
SELECT i.country, p.year, COUNT(DISTINCT r.patent_id) AS total_patents
FROM relationships r
JOIN inventors i ON r.inventor_id = i.inventor_id
JOIN patents p ON r.patent_id = p.patent_id
WHERE i.country IS NOT NULL
  AND i.country != 'Unknown'
  AND p.year IS NOT NULL
GROUP BY i.country, p.year
ORDER BY p.year, total_patents DESC;
"""

country_trends = pd.read_sql(q, conn)
print(country_trends.head(20))
country_trends.to_csv("../output/country_trends.csv", index=False)
print("country_trends.csv saved")

   country  year  total_patents
0       US  1976          45544
1       JP  1976           5256
2       DE  1976           4613
3       FR  1976           2525
4       CH  1976           1664
5       GB  1976           1652
6       CA  1976           1417
7       NL  1976            803
8       IT  1976            791
9       SE  1976            445
10      BE  1976            373
11      AU  1976            356
12      AT  1976            355
13      RU  1976            325
14      DK  1976            193
15      IL  1976            149
16      ES  1976            116
17      NO  1976            108
18      FI  1976            100
19      CZ  1976             95
country_trends.csv saved


In [13]:
import json

report_data = {
    "total_patents": int(pd.read_sql("SELECT COUNT(1) AS total FROM patents;", conn).iloc[0, 0]),
    "top_inventors": top_inventors.to_dict(orient="records"),
    "top_companies": top_companies.to_dict(orient="records"),
    "top_countries": top_countries.to_dict(orient="records")
}

with open("../output/report.json", "w") as f:
    json.dump(report_data, f, indent=4)

print("json report saved")

json report saved


In [14]:
#console report
total_patents = int(pd.read_sql("SELECT COUNT(1) AS total FROM patents;", conn).iloc[0, 0])

print("=" * 50)
print("PATENT REPORT")
print("=" * 50)
print(f"Total Patents: {total_patents}")

print("\nTop Inventors:")
for i, row in top_inventors.iterrows():
    print(f"{i+1}. {row['name']} - {row['total_patents']}")

print("\nTop Companies:")
for i, row in top_companies.iterrows():
    print(f"{i+1}. {row['name']} - {row['total_patents']}")

print("\nTop Countries:")
for i, row in top_countries.iterrows():
    print(f"{i+1}. {row['country']} - {row['total']}")

PATENT REPORT
Total Patents: 9454161

Top Inventors:
1. Kia Silverbrook - 4778
2. Shunpei Yamazaki - 6787
3. Bartley K. Andre - 2478
4. Frederick E. Shelton, IV - 2723
5. Duncan Robert Kerr - 2534
6. Kangguo Cheng - 2835
7. Tao Luo - 4490
8. Peter Gaal - 2554
9. Junyi Li - 2881
10. Jonathan P. Ive - 2947

Top Companies:
1. CANON KABUSHIKI KAISHA - 91328
2. HITACHI, LTD. - 46073
3. Intel Corporation - 53218
4. LG ELECTRONICS INC. - 50000
5. SONY GROUP CORPORATION - 62908
6. International Business Machines Corporation - 164081
7. SAMSUNG DISPLAY CO., LTD. - 174536
8. General Electric Company - 52040
9. Fujitsu Limited - 56341
10. Kabushiki Kaisha Toshiba - 53604

Top Countries:
1. US - 1987483
2. JP - 499038
3. DE - 302484
4. CN - 196441
5. KR - 154040
6. FR - 142006
7. CA - 113916
8. GB - 112701
9. TW - 100718
10. IN - 74851


In [15]:
conn.close()
print("db connection closed")

db connection closed
